# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary task: ranking / scoring in a hybrid ML pipeline.**

The system must answer: **which content pages should get attention first, why, and what action should be considered?** That makes ranking/scoring the final task.

The ranking is built in stages:

1. **Signal analysis** creates a leakage-safe feature set from observed search, traffic, engagement, freshness, and content data.
2. **Clustering** groups similar pages into performance archetypes and gives each page a fair peer group.
3. **Peer-relative analysis** measures how unusual a page is compared with similar pages, for example whether its CTR or engagement is weak for its archetype.
4. **Classification / prediction** estimates a future observed performance state using only information available before the decision point.
5. **Impact estimation** combines predicted risk, expected size of the change, and page exposure/value.
6. **Ranking / scoring** orders pages by expected adverse impact.
7. **Action generation** uses the archetype, predicted direction, and strongest abnormal signals to produce reason codes and a suggested action.

In short:

**observed signals → archetype → relative abnormalities → future prediction → expected impact → ranked priority → suggested action**

The suggested action is an evidence-based **intervention hypothesis**, not a proven causal effect. We can test prediction quality retrospectively and later test whether the recommended action causes improvement with a controlled or otherwise valid causal study.

In [ ]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


## 2. Target or proxy

Different stages use different learning objects.

### Clustering

Clustering has **no target column**. It creates performance archetypes and peer baselines from the training data.

### Supervised prediction

The main supervised target should be an **observed future outcome**, not a manually created priority score or action label.

The intended structure is:

**past feature window → decision point → non-overlapping future outcome window**

The later warehouse target will be a future decline outcome such as `future_decline_30d`, built from observed search performance after the decision point. A continuous future-change value will also keep the size of the movement, so a small decline and a severe decline are not treated as equal.

The exact decline threshold, persistence rule, and minimum-volume floor are **not fixed in this notebook**. They will be defined in the data-contract stage and tested for sensitivity before model training.

### Starter-data proxy

The 30,000-row starter CSV is only a trailing-90-day snapshot, so it cannot provide a true future target. For Assignment 3, the transparent proxy is:

[
\text{decline\_proxy}=1
\quad \text{if} \quad
\texttt{trend\_direction = "down"}
]

This is a **defined current-window proxy**, not a future ground-truth outcome and not proof that a page needs intervention.

Because `trend_direction` comes from `trend_pct`, neither field can be used as a predictive feature when this proxy is the target.

### Ranking and action

The final ranking will not learn a hand-written priority label. Conceptually:

[
\text{priority}
\propto
P(\text{future decline})
\times
E(\text{decline magnitude})
\times
\text{measured exposure/value}
]

The exact scaling will be fixed later and validated rather than assigned arbitrary weights.

The action stage maps the page's archetype, predicted future state, and strongest peer-relative abnormalities to a **recommended intervention hypothesis** that can later be tested.

In [ ]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.